# jaxfne Suite No. 1: Computational Biophysics

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HNXJ/jaxfne/blob/main/tutorials/jaxfne_suite_no_1_computational_biophysics.ipynb)

This tutorial shows how to configure, simulate, visualize, and inspect neural population models with the JAX-native `jaxfne` workflow:

`Configuration → construct → simulate → probe/visualize`

The examples use simulated/proxy-scale readouts inside a computational scaffold.

## Learning Objectives

1. Configure and simulate single-neuron Izhikevich dynamics with explicit runtime settings.
2. Build excitatory/inhibitory population models using the public configuration API.
3. Generate source/field/probe-style proxy readouts for laminar or population-level interpretation.
4. Run compact optimization or validation summaries without manual trial-and-error tuning.

**Prerequisites:** `jaxfne`, `jax`, `matplotlib`

**Runtime target:** 3–5 minutes on CPU or Colab.


In [ ]:
# Install jaxfne (if needed)
try:
    import jaxfne
except ImportError:
    !pip install jaxfne -q

import jaxfne as jtfne
import numpy as np
import matplotlib.pyplot as plt
import json
from pathlib import Path
from dataclasses import dataclass
from typing import Optional, List, Dict, Any

## Setup


In [ ]:
# Runtime configuration
DURATION_MS = 5000.0
DT_MS = 0.1
SEED = 42
DTYPE = "float32"

# Figure output directory
FIG_DIR = Path("figures") / "suite_no1"
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Public metadata (scope status, readout mode, field approximation)
RUN_METADATA = {
    "scope_status": "computational_scaffold",
    "readout_status": "simulated_proxy",
    "field_mode": "proxy_convolution_no_pde",
    "physical_amplitude_claim_allowed": False,
    "duration_ms": DURATION_MS,
    "dt_ms": DT_MS,
    "dtype": DTYPE,
    "seed": SEED,
}

# Ensure all values are JSON-safe
json.dumps(RUN_METADATA, allow_nan=False)
print("Run metadata initialized (JSON-safe).")


## Setup (continued)


In [ ]:
def save_png(fig, name: str) -> str:
    """Save figure to PNG and return path."""
    path = FIG_DIR / f"{name}.png"
    fig.savefig(path, dpi=150, bbox_inches="tight")
    plt.close(fig)
    size_kb = path.stat().st_size / 1024.0
    print(f"saved: {path} ({size_kb:.1f} KB)")
    return str(path)

def finite_status(*arrays) -> bool:
    """Check if all arrays contain only finite values."""
    return all(bool(np.all(np.isfinite(np.asarray(a)))) 
               for a in arrays if a is not None)

def population_rate_hz(spikes: np.ndarray, dt_ms: float) -> float:
    """Compute mean population firing rate in Hz."""
    spikes = np.asarray(spikes)
    if spikes.size == 0:
        return 0.0
    return float(spikes.mean() * (1000.0 / dt_ms))

@dataclass
class ConfigSummary:
    """Summary of configured model."""
    name: str
    n_neurons: int
    layers: List[str]
    cell_types: Dict[str, float]
    connectivity: str
    emitter_family: str
    emitter_preset: str
    probes: List[str]

def cfg_summary_rows(cfg: 'jtfne.Configuration', name: str) -> ConfigSummary:
    """Extract configuration summary for display."""
    # Extract from configuration metadata
    cfg_dict = cfg._config if hasattr(cfg, '_config') else {}
    
    return ConfigSummary(
        name=name,
        n_neurons=cfg_dict.get('n', '?'),
        layers=cfg_dict.get('layers', ['L2/3']),
        cell_types=cfg_dict.get('cell_types', {}),
        connectivity=cfg_dict.get('connectivity', 'unknown'),
        emitter_family=cfg_dict.get('emitter_family', 'izhikevich'),
        emitter_preset=cfg_dict.get('emitter_preset', 'cortical_eig'),
        probes=cfg_dict.get('probes', []),
    )

def display_run_summary(label: str, spikes: np.ndarray, V_m: np.ndarray, 
                       dt_ms: float, finite: bool):
    """Display simulation summary as a clean table."""
    rate_hz = population_rate_hz(spikes, dt_ms)
    print(f"\n{label}:")
    print(f"  Spikes: {int(spikes.sum())} | Shape: {spikes.shape}")
    print(f"  Mean rate: {rate_hz:.2f} Hz")
    print(f"  Voltage range: [{V_m.min():.1f}, {V_m.max():.1f}] mV-like")
    print(f"  All finite: {finite}")

# Visualization helper functions
def plot_raster(spike_times_list, spike_neuron_ids_list, t, figsize=(10, 4), title="Population Raster"):
    """Plot spike raster from list of spike times per neuron."""
    fig, ax = plt.subplots(figsize=figsize)
    for neuron_id, (times, ids) in enumerate(zip(spike_times_list, spike_neuron_ids_list)):
        ax.scatter(times, ids, s=2, alpha=0.6)
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Neuron index")
    ax.set_title(title)
    ax.set_xlim(t.min(), t.max())
    return fig

def plot_population_rate(t, spikes, bin_ms=25.0, figsize=(10, 3), title="Population Rate"):
    """Plot time-binned population firing rate."""
    bin_edges = np.arange(0, t.max() + bin_ms, bin_ms)
    rates = []
    for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
        mask = (t >= lo) & (t < hi)
        rates.append(float(spikes[mask].mean() * (1000.0 / DT_MS)))
    
    fig, ax = plt.subplots(figsize=figsize)
    ax.plot(0.5 * (bin_edges[:-1] + bin_edges[1:]), rates, lw=1.5)
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Mean rate (Hz)")
    ax.set_title(title)
    return fig, rates

def plot_voltage_samples(t, V_m, title="Voltage trajectory", figsize=(10, 3), max_neurons=10):
    """Plot voltage time series from first N neurons."""
    fig, ax = plt.subplots(figsize=figsize)
    n_show = min(max_neurons, V_m.shape[1])
    for i in range(n_show):
        ax.plot(t, V_m[:, i], lw=0.8, alpha=0.7)
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("V-like state (mV)")
    ax.set_title(title)
    return fig

def plot_connectivity_matrix(W, title="Connectivity matrix", figsize=(5, 5)):
    """Plot connectivity weight matrix."""
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(W, aspect="auto", cmap="RdBu", vmin=-W.max(), vmax=W.max())
    ax.set_xlabel("Sending neuron")
    ax.set_ylabel("Receiving neuron")
    ax.set_title(title)
    fig.colorbar(im, ax=ax, fraction=0.046)
    return fig

def plot_laminar_readout(t, lfp_proxy, csd_proxy=None, figsize=(12, 4), title="Laminar Readout"):
    """Plot LFP-proxy and optionally CSD-proxy."""
    if csd_proxy is not None:
        fig, axes = plt.subplots(1, 2, figsize=figsize)
        # LFP
        axes[0].plot(t, lfp_proxy[:, :4], lw=0.8)
        axes[0].set_title("LFP-proxy (first 4 contacts)")
        axes[0].set_xlabel("Time (ms)")
        axes[0].set_ylabel("Proxy units")
        # CSD
        axes[1].imshow(csd_proxy.T, aspect="auto", origin="upper", cmap="RdBu")
        axes[1].set_title("CSD-proxy heatmap")
        axes[1].set_xlabel("Time index")
        axes[1].set_ylabel("Contact")
    else:
        fig, ax = plt.subplots(figsize=figsize)
        ax.plot(t, lfp_proxy[:, :4], lw=0.8)
        ax.set_title("LFP-proxy")
        ax.set_xlabel("Time (ms)")
        ax.set_ylabel("Proxy units")
    
    fig.suptitle(title)
    return fig

print("✓ Display and visualization helpers loaded.")

## Section 3: Mathematical Glossary

### Izhikevich Neuron Model

The Izhikevich model is a **phenomenological** (simplified for teaching, focusing on network dynamics) model of neuronal dynamics:

$$\frac{dv_i}{dt} = 0.04 v_i^2 + 5v_i + 140 - u_i + I_i$$
$$\frac{du_i}{dt} = a_i(b_i v_i - u_i)$$

**When $v \geq 30$ mV:** reset to $c$, increment recovery by $d$.

**Parameters:**
- $v_i$: voltage-like state
- $u_i$: recovery variable  
- $I_i$: input current (combination of base drive + recurrent input)
- $a, b, c, d$: preset-dependent parameters

**Presets** (e.g., "cortical_eig") are pre-tuned parameter sets for common neuron types.

### Recurrent Input Scaffold

Each neuron receives:

$$I_i(t) = I^{\text{base}}(t) + \sum_j W_{ij} s_j(t)$$

Where:
- $I^{\text{base}}$: baseline drive
- $W_{ij}$: synaptic weight (sign determined by E/I type)
- $s_j(t)$: spike/activity signal from neuron $j$

### Proxy Readout Principle

Field quantities (LFP, CSD, etc.) are computed via **convolution-based proxies**, not PDE solvers:

$$\text{LFP-proxy}(t) = \text{convolve}(\text{sources}(t), \text{kernel})$$

This allows fast visualization without claiming physical amplitude calibration.

### Scope Boundary

All outputs in this tutorial are:
- **Computational scaffold**: a teaching model, tutorial-scale for learning and visualization
- **Proxy-scale**: designed for relative comparison, not absolute measurement
- **Phenomenological**: not mechanistically detailed biophysics


## Part 1: Single-Neuron Dynamics

**Goal:** simulate one configured emitter and inspect its state traces.

Main objects:

- `Configuration`
- emitter preset
- runtime settings
- simulated voltage/state output


In [ ]:
# Part 1: Single Neuron
# Configuration is the entry point for all models, including single neurons.

cfg_single = (jtfne.Configuration()
    .runtime(seed=SEED, dtype=DTYPE, duration_ms=DURATION_MS, dt_ms=DT_MS)
    .column(name="single_neuron", layers=["L2/3"], n=1)
    .cell_types({"E": 1.0})
    .connectivity(kind="none")  # Single neuron: no recurrence
    .set_emitter("izhikevich", "cortical_eig")
    .probes(["spikes", "V_m", "source", "LFP-proxy"]))

# Validation
is_valid = cfg_single.validate()["valid"]
print(f"Single-neuron config valid: {is_valid}")

# Construct and simulate
model_single = jtfne.construct(cfg_single)
signals_single = jtfne.simulate(
    model_single, 
    duration_ms=DURATION_MS, 
    dt_ms=DT_MS, 
    seed=SEED
)

# Access readouts directly from signals
single_spikes = np.asarray(signals_single.spikes)[:, 0]
single_V_m = np.asarray(signals_single.V_m)[:, 0]
single_sources = np.asarray(signals_single.sources)[:, 0]
single_t = np.asarray(signals_single.time_ms)
single_rate_hz = population_rate_hz(single_spikes[:, None], DT_MS)

# Display summary
display_run_summary(
    "Single Neuron Summary",
    single_spikes[:, None],
    single_V_m[:, None],
    DT_MS,
    finite_status(single_V_m, single_spikes, single_sources)
)

# Figures
fig = plot_voltage_samples(single_t, single_V_m[:, None], 
    title="Figure 1a — Single Neuron Voltage Trajectory", max_neurons=1)
save_png(fig, "01_single_voltage_trajectory")

fig = plt.figure(figsize=(10, 3))
ax = fig.add_subplot(111)
ax.plot(single_t, single_sources, lw=1.0, color="steelblue")
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Source proxy")
ax.set_title("Figure 1b — Single Neuron Source Proxy")
save_png(fig, "02_single_source_proxy")


## Part 2: E/I Population Dynamics

**Goal:** scale from one emitter to an excitatory/inhibitory population.

This section demonstrates how population composition, connectivity, and runtime settings shape simulated spike and state trajectories.


In [ ]:
# Part 2: E/PV Population
# Same Configuration grammar, now with declared E/I split and connectivity.

cfg_pop = (jtfne.Configuration()
    .runtime(seed=SEED + 1, dtype=DTYPE, duration_ms=DURATION_MS, dt_ms=DT_MS)
    .column(name="small_epv_pop", layers=["L2/3"], n=16)
    .cell_types({"E": 0.75, "PV": 0.25})
    .connectivity(kind="dense_signed_ei_metadata", e_to_all=0.08, i_to_all=-0.12)
    .set_emitter("izhikevich", "cortical_eig")
    .probes(["spikes", "V_m", "source", "LFP-proxy"]))

# Construct and simulate
model_pop = jtfne.construct(cfg_pop)
signals_pop = jtfne.simulate(
    model_pop,
    duration_ms=DURATION_MS,
    dt_ms=DT_MS,
    seed=SEED + 1
)

# Readouts from signals
pop_spikes = np.asarray(signals_pop.spikes)
pop_V_m = np.asarray(signals_pop.V_m)
pop_sources = np.asarray(signals_pop.sources)
pop_t = np.asarray(signals_pop.time_ms)
pop_rate_hz = population_rate_hz(pop_spikes, DT_MS)

# Display summary
display_run_summary(
    "E/PV Population Summary",
    pop_spikes,
    pop_V_m,
    DT_MS,
    finite_status(pop_V_m, pop_spikes, pop_sources)
)

# Figures
fig, ax = plt.subplots(figsize=(10, 4))
for neuron_id in range(pop_spikes.shape[1]):
    spike_times = pop_t[pop_spikes[:, neuron_id] > 0.5]
    ax.scatter(spike_times, np.full_like(spike_times, neuron_id), s=3, alpha=0.7)
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Neuron index")
ax.set_title("Figure 2a — Configured E/PV Population Raster")
ax.set_xlim(pop_t.min(), pop_t.max())
save_png(fig, "03_epv_population_raster")

fig, rates = plot_population_rate(pop_t, pop_spikes, bin_ms=50.0,
    figsize=(10, 3), title="Figure 2b — Configured E/PV Population Rate")
save_png(fig, "04_epv_population_rate")

# Connectivity (from model parameters)
W = np.asarray(model_pop.params["emitter"].W) if "emitter" in model_pop.params else None
if W is not None:
    fig = plot_connectivity_matrix(W, title="Figure 2c — E/PV Connectivity Matrix")
    save_png(fig, "05_epv_connectivity_matrix")


## Part 3: Source/Field/Probe-Style Readouts

**Goal:** map simulated neural activity into proxy readouts that can be visualized across population or laminar structure.

These readouts are computational proxies. They support tutorial interpretation, not calibrated physical amplitude claims.


In [ ]:
# Part 3: Laminar Column
# Extended configuration with multiple layers and contacts.

cfg_column = (jtfne.Configuration()
    .runtime(seed=SEED + 2, dtype=DTYPE, duration_ms=DURATION_MS, dt_ms=DT_MS)
    .column(name="laminar_column", layers=["L2/3", "L4", "L5", "L6"], n=48)
    .cell_types({"E": 0.75, "PV": 0.12, "SST": 0.08, "VIP": 0.05})
    .connectivity(kind="laminar_signed_metadata", recurrent=True)
    .set_emitter("izhikevich", "cortical_eig")
    .probes(["spikes", "V_m", "source", "LFP-proxy", "CSD-proxy"], n_contacts=16))

# Construct and simulate
model_column = jtfne.construct(cfg_column)
signals_column = jtfne.simulate(
    model_column,
    duration_ms=DURATION_MS,
    dt_ms=DT_MS,
    seed=SEED + 2
)

# Readouts
column_spikes = np.asarray(signals_column.spikes)
column_V_m = np.asarray(signals_column.V_m)
column_sources = np.asarray(signals_column.sources)
column_t = np.asarray(signals_column.time_ms)
column_rate_hz = population_rate_hz(column_spikes, DT_MS)

# Field readouts via probe (for field quantities not in signals)
readouts_column = model_column.probe(signals_column, modes=["LFP-proxy", "CSD-proxy"])
column_lfp_proxy = np.asarray(readouts_column.get("LFP-proxy", np.zeros((len(column_t), 16))))
column_csd_proxy = np.asarray(readouts_column.get("CSD-proxy", np.zeros((len(column_t), 16))))

# Display summary
display_run_summary(
    "Laminar Column Summary",
    column_spikes,
    column_V_m,
    DT_MS,
    finite_status(column_V_m, column_spikes, column_sources, column_lfp_proxy)
)

# Figures
fig, ax = plt.subplots(figsize=(10, 5))
for neuron_id in range(column_spikes.shape[1]):
    spike_times = column_t[column_spikes[:, neuron_id] > 0.5]
    ax.scatter(spike_times, np.full_like(spike_times, neuron_id), s=2, alpha=0.5)
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Neuron index")
ax.set_title("Figure 3a — Laminar Column Population Raster")
ax.set_xlim(column_t.min(), column_t.max())
save_png(fig, "06_laminar_population_raster")

# Package visualization if available
try:
    fig = jtfne.vis.spectrolaminar(signals_column, figsize=(13, 4))
    save_png(fig, "07_spectrolaminar_visualization")
except Exception as e:
    print(f"Note: spectrolaminar visualization not available: {e}")
    # Fallback: manual LFP/CSD plot
    fig = plot_laminar_readout(column_t, column_lfp_proxy, column_csd_proxy,
        figsize=(12, 4), title="Figure 3b — Laminar LFP/CSD Readouts")
    save_png(fig, "07_laminar_lfp_csd_readout")


## Part 4: Compact Tuning or Validation Loop

**Goal:** evaluate model settings against a declared target summary.

The optimization/validation result is a computational score over simulated outputs, not a mechanism proof.


In [ ]:
# Part 4: Objective and Optimization
# Define a simple objective and run a short tuning sweep.

objective = (jtfne.Objective()
    .loss("rate_target", metric="spike_rate_hz_mean", target=10.0, weight=1.0)
    .gate("finite_check", metric="finite_outputs", threshold=True, criterion="equal"))

optimizer = jtfne.random_search(metadata={"section": "suite_no1_tuning"})

# Short simulation for tuning (avoid long compute)
short_sim = jtfne.Simulation(duration_ms=300.0, dt_ms=DT_MS, seed=SEED + 3)

# Tune one parameter
try:
    model_tuned, tuning_report_raw = model_column.tune(
        objective,
        optimizer=optimizer,
        steps=3,
        seed=SEED + 3,
        simulation=short_sim,
        parameter="source_scale",
        bounds=(0.5, 1.5),
    )
    tuning_succeeded = True
except Exception as e:
    print(f"Tuning note: {e}")
    tuning_report_raw = {"tuning_status": "skipped", "reason": str(e)}
    tuning_succeeded = False

# Public-facing tuning report (scope status only)
tuning_report = {
    "tuning_status": tuning_report_raw.get("tuning_status", "completed"),
    "steps_requested": tuning_report_raw.get("steps_requested", 3),
    "scope_status": "computational_scaffold",
    "readout_status": "proxy_scale",
}

json.dumps(tuning_report, allow_nan=False)
print("Tuning metadata (public):")
print(json.dumps(tuning_report, indent=2))

# Visualize tuning history if available
if tuning_succeeded and "candidate_history" in tuning_report_raw:
    history = tuning_report_raw["candidate_history"]
    if history:
        steps = [h.get("step", i) for i, h in enumerate(history)]
        scores = [h.get("score", np.nan) for h in history]
        fig, ax = plt.subplots(figsize=(7, 3))
        ax.plot(steps, scores, marker="o", lw=1.5)
        ax.set_xlabel("Candidate step")
        ax.set_ylabel("Objective score")
        ax.set_title("Figure 4 — Tuning Objective Score History")
        save_png(fig, "08_tuning_objective_history")
    else:
        print("No tuning history available.")
else:
    print("Tuning completed without candidate history.")


## Validation & Artifacts


In [ ]:
# Gather all run metadata and validation checks
figure_files = sorted(str(p) for p in FIG_DIR.glob("*.png"))

validation_manifest = {
    **RUN_METADATA,
    "models_configured": ["single_neuron", "e_pv_population", "laminar_column"],
    "public_api_path": [
        "jtfne.Configuration",
        "cfg.runtime", "cfg.column", "cfg.cell_types", "cfg.connectivity",
        "cfg.set_emitter", "cfg.probes",
        "jtfne.construct", "jtfne.simulate",
        "signals.spikes", "signals.V_m", "signals.sources", "signals.time_ms",
        "model.probe",
        "jtfne.vis.spectrolaminar",
        "jtfne.Objective", "jtfne.random_search", "model.tune",
    ],
    "n_figures": len(figure_files),
    "figures": figure_files,
    "outputs_finite": {
        "single_neuron": finite_status(single_V_m, single_spikes),
        "e_pv_population": finite_status(pop_V_m, pop_spikes),
        "laminar_column": finite_status(column_V_m, column_spikes),
    },
    "population_rates_hz": {
        "single_neuron": round(single_rate_hz, 4),
        "e_pv_population": round(pop_rate_hz, 4),
        "laminar_column": round(column_rate_hz, 4),
    },
}

# Write manifest
manifest_path = Path("suite_no1_public_manifest.json")
manifest_path.write_text(json.dumps(validation_manifest, indent=2), encoding="utf-8")

print("\nValidation Manifest Summary:")
print(f"  Figures generated: {validation_manifest['n_figures']}")
print(f"  All outputs finite: {all(validation_manifest['outputs_finite'].values())}")
print(f"  Population rates (Hz):")
for model, rate in validation_manifest['population_rates_hz'].items():
    print(f"    - {model}: {rate}")
print(f"  Manifest saved: {manifest_path}")


## Section 9: Export Artifacts

In [ ]:
import shutil

# Ensure notebook outputs are not committed
notebook_outputs_dir = Path("tutorial_outputs")
if notebook_outputs_dir.exists():
    print(f"Note: {notebook_outputs_dir} exists; not committing.")

# Archive figures for external review (optional)
if FIG_DIR.exists() and list(FIG_DIR.glob("*.png")):
    archive_name = "suite_no1_figures"
    archive_path = shutil.make_archive(
        archive_name,
        "zip",
        FIG_DIR.parent,
        FIG_DIR.name
    )
    print(f"Figures archived: {archive_path}")
else:
    print("No figures to archive.")

print("\n✓ All outputs saved. Ready for interpretation and validation.")


## Summary

Completed workflow:

`Configuration → construct → simulate → probe/visualize → summarize`

What this tutorial demonstrates:

- package-native configuration grammar
- single-neuron and population simulation
- simulated/proxy readouts
- compact score or validation reporting

**Scope boundary:**

The outputs are tutorial-scale computational readouts. Physical calibration, subject-specific geometry, full volume-conductor solving, and empirical validation require separate evidence.


## Exercises

1. Vary E/I ratio: Change `cfg_pop.cell_types(...)` and re-run Part 2.
2. Explore presets: Try `set_emitter("izhikevich", "fast_spiking")` instead of "cortical_eig".
3. Extended simulation: Increase `DURATION_MS` to 10,000 and observe long-timescale dynamics.
4. Multi-layer connectivity: Set `kind="inter_laminar_signed"` in Part 3 and inspect layer-wise rates.

---

**For more:** See [API Reference](../api/index.md), [Guides](../guides/index.md), and [Calibration](../guides/calibration.md).
